<a href="https://colab.research.google.com/github/mesuvein23/SocrateAI/blob/main/agent2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [27]:
!pip install -q google-adk python-dotenv

In [28]:
import os
from google.colab import userdata
os.environ["GOOGLE_API_KEY"] = userdata.get("agentkey")
os.environ["MODEL"] = "gemini-flash-lite-latest"

In [29]:
import os
import sys
from pathlib import Path
import datetime

from dotenv import load_dotenv
from google.adk.agents import Agent, LoopAgent
from google.adk.tools import agent_tool

# ── env/config ───────────────────────────────────────────────────────────────
load_dotenv()

MODEL = os.getenv("MODEL", "gemini-flash-latest")

# ── Sub-Agent: Planner ───────────────────────────────────────────────────────
blog_planner = Agent(
   name="BlogPlanner",
   model=MODEL,
   description="Creates a practical, skimmable outline in Markdown.",
   instruction="""
You are a technical content strategist. Produce a clear Markdown outline with:
- Title
- Short intro
- 4–6 main sections (each with 2–3 bullets)
- Conclusion

If `codebase_context` exists in state, weave in specific sections/snippets.
Return only the outline in Markdown.
""",
   output_key="blog_outline",
)

class OutlineValidationChecker(Agent):
   def __init__(self):
       super().__init__(
           name="OutlineValidationChecker",
           model=MODEL,
           description="Validates that the outline is usable.",
           instruction="""
Check the outline in state `blog_outline`. If it has a title, intro, 4–6 sections, and a conclusion, respond exactly "ok".
Otherwise respond exactly "retry" and list missing pieces.
""",
           output_key="validation_result",
       )

robust_blog_planner = LoopAgent(
   name="RobustBlogPlanner",
   description="Retries planning if validation fails.",
   sub_agents=[blog_planner, OutlineValidationChecker()],
   max_iterations=3,
)

# ── Sub-Agent: Writer ────────────────────────────────────────────────────────
blog_writer = Agent(
   name="BlogWriter",
   model=MODEL,
   description="Writes a technical blog post from the outline.",
   instruction="""
Write a complete Markdown article from the outline in `blog_outline`.

Guidelines:
- Audience: software engineers; skip basics and focus on practical insight.
- Explain both the 'how' and 'why'.
- Include concise code snippets when helpful.
- Follow the outline’s structure (H2/H3).
- Output only the final article in Markdown (no fence around the whole post).
""",
   output_key="blog_post",
)

class BlogPostValidationChecker(Agent):
   def __init__(self):
       super().__init__(
           name="BlogPostValidationChecker",
           model=MODEL,
           description="Validates the final post.",
           instruction="""
Check `blog_post` for: intro, clear sections matching the outline, conclusion, and technical clarity.
If passes, respond "ok". Else respond "retry" with the specific fixes.
""",
           output_key="validation_result",
       )

robust_blog_writer = LoopAgent(
   name="RobustBlogWriter",
   description="Retries writing if validation fails.",
   sub_agents=[blog_writer, BlogPostValidationChecker()],
   max_iterations=3,
)

# Expose planner/writer as tools so the root agent can call them explicitly
planner_tool = agent_tool.AgentTool(agent=robust_blog_planner)
writer_tool  = agent_tool.AgentTool(agent=robust_blog_writer)

# ── Root Agent: Plan → Write ────────────────────────────────────────────────
root_agent = Agent(
   name="Blogger",
   model=MODEL,
   description="Minimal multi-agent blogger that plans and writes.",
   instruction=f"""
If the user gives a topic:
1) Call the planner tool to generate the outline.
2) Call the writer tool to produce the full draft.
3) End with 3 alternate titles and 2 tweet-length hooks.

Date: {datetime.datetime.now().strftime("%Y-%m-%d")}
""",
   tools=[
       planner_tool, # calls RobustBlogPlanner
       writer_tool,  # calls RobustBlogWriter
   ],
)

/tmp/ipykernel_1695/707610438.py:46: DeprecationWarning: LoopAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  robust_blog_planner = LoopAgent(
/tmp/ipykernel_1695/707610438.py:84: DeprecationWarning: LoopAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  robust_blog_writer = LoopAgent(


In [30]:
from google.adk.runners import InMemoryRunner

runner = InMemoryRunner(agent=root_agent)
await runner.run_debug("Nepal For Travelling")

Blogger > # Discovering the Roof of the World: The Ultimate Travel Guide to Nepal

Nestled like a crown jewel in the folds of the Himalayas, Nepal is a destination that captures the imagination of every traveler. Whether you are an adrenaline junkie looking to conquer high-altitude passes, a spiritual seeker searching for inner peace, or a nature lover eager to spot rhinos and tigers in the lowlands, Nepal offers an extraordinary tapestry of experiences. 

If you are planning your next great adventure, here is why Nepal should be at the very top of your bucket list, along with everything you need to know to make the journey of a lifetime.

---

## 1. Why Nepal? The Land of Contrasts
Nepal is geographically compact, yet it packs an astonishing variety of landscapes and cultures into its borders. Within a single day's journey, you can transition from subtropical jungles and raging rivers to sub-alpine forests and towering, snow-capped giants. 

Home to eight of the world’s fourteen peaks

[Event(model_version='gemini-3.5-flash-lite', content=Content(
   parts=[
     Part(
       function_call=FunctionCall(
         args={
           'request': 'Nepal For Travelling'
         },
         id='call_5238717705524731448',
         name='RobustBlogPlanner'
       ),
       thought_signature=b'\x12\x96\x01\n\x93\x01\x01X\xd5\x07{\x91\x0b\xcc\xc5\x15r1y\x93\x15\xbb\x13\xe6F\xf2\x10\xaf\x13\xaf\xe6A\x82\t"\x82\x14\xc9>\xc3\x94\xf93\x95\x03\x1a\xcd[\x13\x80g\xd4\x1a\xa4bpv\xdd\xfc`\x05|\xbf\x1e\xa3\xec\xae\xad8\xac\x16\xb9 \xb7\xfd\x95\xa2\xfc\xd1IW\x9c\xefd\xb5\x1dB\xfd2y\xc5dG\x1d\xdb\xfd!...'
     ),
   ],
   role='model'
 ), grounding_metadata=None, partial=None, turn_complete=None, turn_complete_reason=None, finish_reason=<FinishReason.STOP: 'STOP'>, error_code=None, error_message=None, interrupted=None, custom_metadata=None, usage_metadata=GenerateContentResponseUsageMetadata(
   candidates_token_count=18,
   prompt_token_count=187,
   prompt_tokens_details=[
     ModalityT